# EDA Part 3 (ml.m5.12xlarge)

In [1]:
import os
import pandas as pd
import numpy as np
import functions as func
import json

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')
str_dirname_output = './output'

str_id = 'uniqueid'
str_datecol = 'applicationdate__app'
str_target = 'target'

list_cols_id = [
    str_id,
    str_datecol,
    str_target,
    'data_set',
    'year_month',
]

Project: 20231010-gen-xii
Step: 06_eda_pt_3


### Output

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Iterate through data sets

In [4]:
%%time

# iterate through data sets
for str_df in ['train','valid','test']:
    # print
    print(f'Data set: {str_df}')
    # import data
    str_filename = f'df_{str_df}_noleaks.gzip'
    str_uri = f's3://{str_project}/01_ad/01_data_prep/05_leaky_features/04_write_dfs/{str_filename}'
    # read from s3
    df = pd.read_parquet(str_uri)
    # replace
    df.replace(['NaN','nan'], np.nan, inplace=True)
    
    # get df info
    str_filename = f'dict_df_info_{str_df}.json'
    func.get_df_info(
        df=df, 
        str_datecol=str_datecol, 
        str_dirname_output=str_dirname_output, 
        str_filename=str_filename,
    )
    
    # get descriptives
    str_filename = f'df_descriptives_{str_df}.csv'
    list_cols = [col for col in df.columns if col not in list_cols_id]
    df_tmp = func.get_descriptives_by_column(
        df=df[list_cols], 
        str_dirname_output=str_dirname_output, 
        str_filename=str_filename,
    )
    del df_tmp
    
    # get descriptives of ID cols
    str_filename = f'df_descriptives_id_{str_df}.csv'
    df_tmp = func.get_descriptives_by_column(
        df=df[list_cols_id], 
        str_dirname_output=str_dirname_output, 
        str_filename=str_filename,
    )
    del df_tmp
    
    # plot proportion NaN
    str_filename = f'plt_prop_nan_{str_df}.png'
    func.plot_proportion_nan(
        df=df,
        str_dirname_output=str_dirname_output,
        str_filename=str_filename,
    )
    
    # plot data type frequency
    str_filename = f'plt_dtype_{str_df}.png'
    func.plot_data_type_frequency(
        df=df,
        str_dirname_output=str_dirname_output,
        str_filename=str_filename,
    )
    
    # plot target
    str_filename = f'plt_target_{str_df}.png'
    func.plot_target(
        ser_target=df[str_target],
        str_dirname_output=str_dirname_output,
        str_filename=str_filename,
    )
    print('')

Data set: train


100%|██████████| 1566/1566 [00:00<00:00, 20788.72it/s]



Data set: valid


100%|██████████| 1566/1566 [00:00<00:00, 20372.89it/s]



Data set: test


100%|██████████| 1566/1566 [00:00<00:00, 19851.18it/s]



CPU times: user 7min, sys: 3min 29s, total: 10min 30s
Wall time: 5min 46s


## Create data frame information summary table

In [5]:
# create table
func.get_df_info_summary_table(
    dict_df_info_train=json.load(open(f'{str_dirname_output}/dict_df_info_train.json')), 
    dict_df_info_valid=json.load(open(f'{str_dirname_output}/dict_df_info_valid.json')), 
    dict_df_info_test=json.load(open(f'{str_dirname_output}/dict_df_info_test.json')),
    str_dirname_output=str_dirname_output,
)

,Data Set,Rows,Columns,Total Obs.,Min. Date,Max. Date,Prop. NaN,Target Mean
0,Train,1260418,1566,1973814588,2013-10-01,2017-10-21,0.090121,0.301576
1,Valid,420140,1566,657939240,2017-10-21,2019-02-08,0.093070,0.333867
2,Test,420140,1566,657939240,2019-02-08,2019-12-31,0.115813,0.363688


## Write ```README.md``` to local drive

In [6]:
%%writefile README.md

[Data frame information](output/df_info_summary.csv)

---

Descriptives by column:
- [Train](output/df_descriptives_train.csv)
- [Valid](output/df_descriptives_valid.csv)
- [Test](output/df_descriptives_test.csv)

---

Descriptives by ```id``` column:
- [Train](output/df_descriptives_ids_train.csv)
- [Valid](output/df_descriptives_ids_valid.csv)
- [Test](output/df_descriptives_ids_test.csv)

---

Percent ```NaN``` overall (train - left, valid - middle, test - right):

<p float="left">
  <img src="./output/plt_prop_nan_train.png" width=33% />
  <img src="./output/plt_prop_nan_valid.png" width=33% /> 
  <img src="./output/plt_prop_nan_test.png" width=33% />
</p>

---

Data type frequency (train - left, valid - middle, test - right):

<p float="left">
  <img src="./output/plt_dtype_train.png" width=33% />
  <img src="./output/plt_dtype_valid.png" width=33% /> 
  <img src="./output/plt_dtype_test.png" width=33% />
</p>

---

Target (train - left, valid - middle, test - right):

<p float="left">
  <img src="./output/plt_target_train.png" width=33% />
  <img src="./output/plt_target_valid.png" width=33% /> 
  <img src="./output/plt_target_test.png" width=33% />
</p>

Overwriting README.md
